# Task 1: Agent Concepts & Mental Model

## 1. Agent vs Chatbot vs Workflow

### **Chatbot**
- **Definition**: A conversational interface that responds to user input with a single response.
- **Flow**: One request → One response.
- **Capabilities**: No tools, no autonomy, no memory beyond the current session context.
- **Example**: Basic ChatGPT interface, FAQ customer service bot.
- **Key Characteristic**: **Passive** — waits for input, gives output, does not act on the world.

### **Workflow**
- **Definition**: A predefined, deterministic sequence of steps that always executes the same way.
- **Flow**: Fixed path — every branch is known before execution.
- **Capabilities**: No LLM reasoning involved; purely programmatic logic (if/else, loops, function calls).
- **Example**: "If temperature > 100°C → send alert email; else → log reading."
- **Key Characteristic**: **Rigid** — the computer follows instructions; no decision-making by an LLM.

### **Agent**
- **Definition**: An LLM that **decides what to do next** based on observations from the world.
- **Flow**: Continuous loop — Reason → Act → Observe → Repeat until task completion.
- **Capabilities**:
  - **Autonomy**: Chooses its own actions based on context.
  - **Tool Use**: Invokes external functions (APIs, databases, file I/O, calculations).
  - **Multi-step Planning**: Breaks complex tasks into smaller steps, solves each, combines results.
  - **Self-Correction**: If a tool fails or returns unexpected results, it can retry with a different approach.
- **Example**: An agent that looks up weather in two cities, compares temperatures, and answers "Which is warmer?"
- **Key Characteristic**: **Active** — decides, acts, observes, decides again.

---

## 2. What Makes Something "Agentic"? (The Four Pillars)

| Trait | What It Means | Why It Matters |
|-------|---------------|----------------|
| **Autonomy** | The model decides the next step based on prior observations, not a one-shot prompt. | Handles tasks it wasn't explicitly programmed for; adapts to novel situations. |
| **Tool Use** | The model invokes external functions to get information it doesn't have internally. | Accesses real-time data, performs calculations, reads/writes files, calls APIs. |
| **Multi-step Planning** | Breaks a complex task into smaller steps, executes each, and synthesizes results. | Solves problems too complex for a single LLM call (e.g., research + analysis + report). |
| **Self-Correction** | Detects failures (tool errors, bad results) and tries alternative approaches. | Reduces failure rates; handles edge cases without human intervention. |

**The Spectrum**: Chatbot (no tools, no loop) ←→ Workflow (deterministic steps) ←→ Full Agent (tools + loop + memory + correction). Most production systems sit somewhere in between.

---

## 3. The ReAct Pattern (Reason → Act → Observe → Repeat)

The **ReAct pattern** is the core cognitive loop that every agent follows. It interleaves **reasoning** (thinking) with **acting** (tool calls) and **observing** (processing results).

### Pseudocode Representation
```python
while task_not_complete:
    # REASON: Think about what to do next
    thought = llm_reason(current_state, available_tools)
    
    # ACT: Execute a tool based on reasoning
    if thought.requires_tool:
        tool_result = execute_tool(thought.tool_name, thought.tool_args)
    
    # OBSERVE: Process the result
    observation = process_result(tool_result)
    current_state.update(observation)
    
    # REPEAT: Loop back to reasoning with new information
```

### Visual Diagram: "Compare Weather in Tokyo vs Paris"

```
USER: "Look up weather in Tokyo and Paris, tell me which is warmer."
                    │
                    ▼
        ┌───────────────────────┐
        │ REASON: "I need       │
        │ weather data for      │
        │ both cities. I'll     │
        │ call the weather tool.│
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ ACT: Call             │
        │ weather_tool("Tokyo") │
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ OBSERVE: Tokyo = 22°C │
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ REASON: "Got Tokyo.   │
        │ Now I need Paris."    │
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ ACT: Call             │
        │ weather_tool("Paris") │
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ OBSERVE: Paris = 18°C │
        └───────────┬───────────┘
                    ▼
        ┌───────────────────────┐
        │ REASON: "Tokyo (22°C) │
        │ > Paris (18°C). Task  │
        │ complete."            │
        └───────────┬───────────┘
                    ▼
        FINAL ANSWER: "Tokyo is warmer than Paris (22°C vs 18°C)."
```

**Key Insight**: The loop continues until the LLM returns a **text-only response** (no `tool_use` blocks), signaling the final answer.

---

## 4. When Is an Agent Overkill? (2–3 Sentences)

An agent is **overkill** when:
- The task is **deterministic** and solvable with a simple script or formula (e.g., "Calculate 15% tax on $200" → just write `0.15 * 200`).
- **No external data or tools are needed** — the LLM can answer directly from its training knowledge (e.g., "Summarize this paragraph" → single prompt suffices).
- The problem requires **no multi-step reasoning or decision-making** — a single well-crafted prompt produces the desired output reliably.

**Rule of Thumb**: If you can solve it with **one prompt + zero tool calls + no looping**, don't build an agent. The added complexity (loop logic, error handling, token costs) isn't justified.

---

## 5. Key Takeaways for Task 1

1. **Agent ≠ Chatbot ≠ Workflow** — The defining difference is *who decides the next action*: human (chatbot), programmer (workflow), or LLM (agent).
2. **ReAct is the universal agent loop** — Reason → Act → Observe → Repeat. Every agent framework (LangGraph, CrewAI, etc.) implements this pattern under the hood.
3. **"Agentic" = Autonomy + Tools + Planning + Correction** — Missing any pillar means it's not a full agent.
4. **Judgment matters** — Knowing when *not* to use an agent is as important as knowing how to build one.

In [1]:
#Task 2
import os
import json
from anthropic import Anthropic

# ============================================================
# SETUP: Initialize Anthropic client (set your API key)
# ============================================================
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))  # or hardcode: api_key="sk-ant-..."

# ============================================================
# TOOL DEFINITIONS (JSON Schemas)
# ============================================================
calculator_tool = {
    "name": "calculator",
    "description": "Performs basic arithmetic (add, subtract) on two numbers. Use when the user asks for math calculations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["add", "subtract"], "description": "The operation to perform"},
            "a": {"type": "number", "description": "First operand"},
            "b": {"type": "number", "description": "Second operand"}
        },
        "required": ["operation", "a", "b"]
    }
}

weather_tool = {
    "name": "get_weather",
    "description": "Returns current temperature for a given city. Use when the user asks about weather. This is a stub - returns fake data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name (e.g., 'Tokyo', 'New York')"}
        },
        "required": ["city"]
    }
}

file_reader_tool = {
    "name": "read_file",
    "description": "Reads the content of a text file from the local filesystem. Use when the user asks to read a file.",
    "input_schema": {
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Relative or absolute path to the file"}
        },
        "required": ["path"]
    }
}

ALL_TOOLS = [calculator_tool, weather_tool, file_reader_tool]

# ============================================================
# TOOL EXECUTION FUNCTIONS (You implement these)
# ============================================================
def execute_calculator(operation: str, a: float, b: float) -> str:
    if operation == "add":
        return f"Result: {a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"Result: {a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

def execute_get_weather(city: str) -> str:
    # Stub: return fake temperatures for demo
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

def execute_read_file(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        return f"File content ({path}):\n{content[:2000]}"  # Truncate long files
    except Exception as e:
        return f"Error reading file: {e}"

TOOL_EXECUTORS = {
    "calculator": lambda args: execute_calculator(args["operation"], args["a"], args["b"]),
    "get_weather": lambda args: execute_get_weather(args["city"]),
    "read_file": lambda args: execute_read_file(args["path"]),
}

# ============================================================
# SINGLE TOOL CALL DEMO: "What's 15 + 27?"
# ============================================================
user_question = "What's 15 + 27?"

# 1. Send request with tools
response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=1024,
    messages=[{"role": "user", "content": user_question}],
    tools=ALL_TOOLS,
    tool_choice={"type": "auto"}
)

print("=== MODEL RESPONSE ===")
for block in response.content:
    print(f"  Type: {block.type}")
    if block.type == "tool_use":
        print(f"  Tool: {block.name}")
        print(f"  Args: {block.input}")
        print(f"  ID:   {block.id}")

# 2. Execute tool(s) manually
tool_results = []
for block in response.content:
    if block.type == "tool_use":
        executor = TOOL_EXECUTORS.get(block.name)
        if executor:
            result = executor(block.input)
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result
            })
            print(f"\n=== TOOL EXECUTED: {block.name} ===")
            print(f"Result: {result}")
        else:
            print(f"\n=== UNKNOWN TOOL: {block.name} ===")

# 3. Send tool result back to model (for final answer)
if tool_results:
    messages = [
        {"role": "user", "content": user_question},
        {"role": "assistant", "content": response.content},
        {"role": "user", "content": tool_results}
    ]
    
    final_response = client.messages.create(
        model="claude-3-5-sonnet-20241022",
        max_tokens=1024,
        messages=messages,
        tools=ALL_TOOLS
    )
    
    print("\n=== FINAL ANSWER ===")
    for block in final_response.content:
        if block.type == "text":
            print(block.text)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
# Task 3 
import os
import json
from anthropic import Anthropic

# ============================================================
# SETUP
# ============================================================
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))  # or hardcode: api_key="sk-ant-..."

# ============================================================
# TOOL DEFINITIONS
# ============================================================
calculator_tool = {
    "name": "calculator",
    "description": "Performs basic arithmetic (add, subtract) on two numbers. Use when the user asks for math calculations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["add", "subtract"], "description": "The operation to perform"},
            "a": {"type": "number", "description": "First operand"},
            "b": {"type": "number", "description": "Second operand"}
        },
        "required": ["operation", "a", "b"]
    }
}

weather_tool = {
    "name": "get_weather",
    "description": "Returns current temperature for a given city. Use when the user asks about weather. This is a stub - returns fake data.",
    "input_schema": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name (e.g., 'Tokyo', 'New York')"}
        },
        "required": ["city"]
    }
}

file_reader_tool = {
    "name": "read_file",
    "description": "Reads the content of a text file from the local filesystem. Use when the user asks to read a file.",
    "input_schema": {
        "type": "object",
        "properties": {
            "path": {"type": "string", "description": "Relative or absolute path to the file"}
        },
        "required": ["path"]
    }
}

ALL_TOOLS = [calculator_tool, weather_tool, file_reader_tool]

# ============================================================
# TOOL EXECUTORS
# ============================================================
def execute_calculator(operation: str, a: float, b: float) -> str:
    if operation == "add":
        return f"Result: {a} + {b} = {a + b}"
    elif operation == "subtract":
        return f"Result: {a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

def execute_get_weather(city: str) -> str:
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25, "london": 15, "berlin": 20}
    temp = fake_temps.get(city.lower(), 20)
    return f"Weather in {city.title()}: {temp}°C"

def execute_read_file(path: str) -> str:
    try:
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        return f"File content ({path}):\n{content[:2000]}"
    except Exception as e:
        return f"Error reading file: {e}"

TOOL_EXECUTORS = {
    "calculator": lambda args: execute_calculator(args["operation"], args["a"], args["b"]),
    "get_weather": lambda args: execute_get_weather(args["city"]),
    "read_file": lambda args: execute_read_file(args["path"]),
}

# ============================================================
# AGENT LOOP FUNCTION (Task 3 Core)
# ============================================================
def run_agent(user_query: str, max_iterations: int = 10) -> str:
    """
    Runs the ReAct agent loop until final answer or max_iterations reached.
    Returns the final text answer.
    """
    messages = [{"role": "user", "content": user_query}]
    iteration = 0
    
    print(f"\n{'='*60}")
    print(f"USER QUERY: {user_query}")
    print(f"{'='*60}\n")
    
    while iteration < max_iterations:
        iteration += 1
        print(f"--- Iteration {iteration} ---")
        
        # 1. SEND: Call LLM
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=4096,
            messages=messages,
            tools=ALL_TOOLS,
            tool_choice={"type": "auto"}
        )
        
        # 2. CHECK: Extract tool_use blocks
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]
        
        # Print reasoning (text blocks)
        for block in text_blocks:
            print(f"  🧠 REASONING: {block.text.strip()}")
        
        # 3. If no tool calls → FINAL ANSWER
        if not tool_blocks:
            final_answer = " ".join(b.text for b in text_blocks)
            print(f"\n  ✅ FINAL ANSWER: {final_answer}")
            print(f"\n{'='*60}")
            print(f"COMPLETED in {iteration} iteration(s)")
            print(f"{'='*60}\n")
            return final_answer
        
        # 4. EXECUTE: Run each tool
        tool_results = []
        for block in tool_blocks:
            print(f"  🔧 TOOL CALL: {block.name}({block.input})")
            executor = TOOL_EXECUTORS.get(block.name)
            if executor:
                result = executor(block.input)
                print(f"  📥 RESULT: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result
                })
            else:
                error_msg = f"Error: Unknown tool '{block.name}'"
                print(f"  ❌ {error_msg}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": error_msg
                })
        
        # 5. APPEND: Assistant message + tool results
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    # Max iterations reached
    error_msg = f"Max iterations ({max_iterations}) reached. Agent stopped."
    print(f"\n  ⚠️  {error_msg}")
    print(f"{'='*60}\n")
    return error_msg

In [ ]:
# Task 4

import os
import json
from datetime import datetime
from anthropic import Anthropic

# ============================================================
# SETUP
# ============================================================
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# ============================================================
# TOOL DEFINITIONS (minimal - for demo)
# ============================================================
calculator_tool = {
    "name": "calculator",
    "description": "Performs basic arithmetic (add, subtract) on two numbers.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["add", "subtract"]},
            "a": {"type": "number"}, "b": {"type": "number"}
        },
        "required": ["operation", "a", "b"]
    }
}

weather_tool = {
    "name": "get_weather",
    "description": "Returns current temperature for a given city (stub).",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
}

ALL_TOOLS = [calculator_tool, weather_tool]

def execute_calculator(operation: str, a: float, b: float) -> str:
    if operation == "add": return f"Result: {a} + {b} = {a + b}"
    if operation == "subtract": return f"Result: {a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

def execute_get_weather(city: str) -> str:
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25}
    return f"Weather in {city.title()}: {fake_temps.get(city.lower(), 20)}°C"

TOOL_EXECUTORS = {
    "calculator": lambda args: execute_calculator(args["operation"], args["a"], args["b"]),
    "get_weather": lambda args: execute_get_weather(args["city"]),
}

# ============================================================
# TASK 4: LOGGING UTILITIES (Core Concept)
# ============================================================
def log_reasoning(text: str, iteration: int):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"[{ts}] [Iter {iteration}] 🧠 REASONING: {text}")

def log_tool_call(name: str, args: dict, iteration: int):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"[{ts}] [Iter {iteration}] 🔧 TOOL CALL: {name}({json.dumps(args)})")

def log_observation(name: str, result: str, iteration: int):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    preview = result[:120] + "..." if len(result) > 120 else result
    print(f"[{ts}] [Iter {iteration}] 📥 OBSERVATION [{name}]: {preview}")

def log_working_memory(wm: dict, iteration: int):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    # Show only key fields, not full message history
    summary = {k: v for k, v in wm.items() if k != "messages"}
    print(f"[{ts}] [Iter {iteration}] 💾 WORKING MEMORY: {json.dumps(summary, default=str)}")

# ============================================================
# TASK 4: AGENT LOOP DEMONSTRATING MEMORY & LOGGING
# ============================================================
def demonstrate_memory_and_logging(user_query: str, max_iterations: int = 5) -> str:
    """
    Isolated Task 4 demo: Shows conversation memory vs working memory
    with structured logging at every step.
    """
    # CONVERSATION MEMORY: Full history sent to LLM
    messages = [{"role": "user", "content": user_query}]
    
    # WORKING MEMORY: Internal state (NOT sent to LLM)
    working_memory = {
        "iteration": 0,
        "tool_call_count": 0,
        "accumulated_results": [],
        "extracted_facts": {},
        "errors": []
    }
    
    print(f"\n{'='*60}")
    print(f"TASK 4 DEMO: Memory & Logging")
    print(f"Query: {user_query}")
    print(f"{'='*60}\n")
    
    while working_memory["iteration"] < max_iterations:
        working_memory["iteration"] += 1
        i = working_memory["iteration"]
        
        # Log working memory state
        log_working_memory(working_memory, i)
        
        # Call LLM with conversation memory only
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            messages=messages,
            tools=ALL_TOOLS,
            tool_choice={"type": "auto"}
        )
        
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]
        
        # Log reasoning
        for block in text_blocks:
            log_reasoning(block.text.strip(), i)
        
        if not tool_blocks:
            final = " ".join(b.text for b in text_blocks)
            print(f"\n✅ FINAL: {final}")
            print(f"\n📊 FINAL WORKING MEMORY STATE:")
            log_working_memory({**working_memory, "final_answer": final}, i)
            return final
        
        # Execute tools with logging
        tool_results = []
        for block in tool_blocks:
            working_memory["tool_call_count"] += 1
            log_tool_call(block.name, block.input, i)
            
            executor = TOOL_EXECUTORS.get(block.name)
            if executor:
                result = executor(block.input)
                log_observation(block.name, result, i)
                
                # Update WORKING MEMORY (internal only)
                working_memory["accumulated_results"].append({
                    "tool": block.name, "args": block.input, "result": result
                })
                
                # Extract structured facts (example)
                if block.name == "get_weather" and "°C" in result:
                    city = block.input.get("city", "unknown")
                    try:
                        temp = int(result.split(": ")[1].replace("°C", ""))
                        working_memory["extracted_facts"][f"{city}_temp"] = temp
                    except: pass
                
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
            else:
                err = f"Unknown tool: {block.name}"
                working_memory["errors"].append(err)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": err})
        
        # Update CONVERSATION MEMORY (sent to LLM next iteration)
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    return "Max iterations reached"

# ============================================================
# RUN DEMO
# ============================================================
demonstrate_memory_and_logging("What's the weather in Tokyo and Paris? Which is warmer?")

In [ ]:
#Task 5

import os
import json
from datetime import datetime
from anthropic import Anthropic

# ============================================================
# SETUP
# ============================================================
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# ============================================================
# TOOL DEFINITIONS (same as before)
# ============================================================
calculator_tool = {
    "name": "calculator",
    "description": "Performs basic arithmetic (add, subtract) on two numbers.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["add", "subtract"]},
            "a": {"type": "number"}, "b": {"type": "number"}
        },
        "required": ["operation", "a", "b"]
    }
}

weather_tool = {
    "name": "get_weather",
    "description": "Returns current temperature for a given city (stub).",
    "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}
}

# Tool that FAILS intentionally (for testing)
broken_tool = {
    "name": "broken_calculator",
    "description": "A calculator that randomly fails. Use for testing error handling.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {"type": "string", "enum": ["add", "subtract"]},
            "a": {"type": "number"}, "b": {"type": "number"}
        },
        "required": ["operation", "a", "b"]
    }
}

ALL_TOOLS = [calculator_tool, weather_tool, broken_tool]
ALLOWED_TOOLS = {"calculator", "get_weather", "broken_calculator"}

# ============================================================
# TOOL EXECUTORS (with intentional failures for demo)
# ============================================================
def execute_calculator(operation: str, a: float, b: float) -> str:
    if operation == "add": return f"Result: {a} + {b} = {a + b}"
    if operation == "subtract": return f"Result: {a} - {b} = {a - b}"
    return f"Error: Unknown operation '{operation}'"

def execute_get_weather(city: str) -> str:
    fake_temps = {"tokyo": 22, "paris": 18, "new york": 25}
    return f"Weather in {city.title()}: {fake_temps.get(city.lower(), 20)}°C"

def execute_broken_calculator(operation: str, a: float, b: float) -> str:
    # INTENTIONAL FAILURE: returns error string instead of raising
    return f"Error: Calculator service unavailable (simulated failure)"

TOOL_EXECUTORS = {
    "calculator": lambda args: execute_calculator(args["operation"], args["a"], args["b"]),
    "get_weather": lambda args: execute_get_weather(args["city"]),
    "broken_calculator": lambda args: execute_broken_calculator(args["operation"], args["a"], args["b"]),
}

# ============================================================
# TASK 5: FAILURE MODE DETECTION & MITIGATION
# ============================================================
def is_error_result(result: str) -> bool:
    """Detect silent errors in tool results"""
    error_markers = ["Error:", "Exception:", "Failed:", "Unavailable", "Timeout"]
    return any(marker in result for marker in error_markers)

def validate_tool_call(name: str, args: dict) -> tuple[bool, str]:
    """Validate tool exists and args match schema (simplified)"""
    if name not in ALLOWED_TOOLS:
        return False, f"Hallucinated tool: '{name}' not in allowed tools: {ALLOWED_TOOLS}"
    
    # Basic arg validation (extend with Pydantic in production)
    if name == "calculator" or name == "broken_calculator":
        if "operation" not in args or args["operation"] not in ["add", "subtract"]:
            return False, f"Invalid operation: {args.get('operation')}. Must be 'add' or 'subtract'"
        if "a" not in args or "b" not in args:
            return False, "Missing required arguments 'a' and/or 'b'"
        if not isinstance(args["a"], (int, float)) or not isinstance(args["b"], (int, float)):
            return False, "Arguments 'a' and 'b' must be numbers"
    
    return True, ""

# ============================================================
# LOGGING
# ============================================================
def log(msg: str, level: str = "INFO", iteration: int = 0):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    prefix = {"INFO": "🧠", "TOOL": "🔧", "OBS": "📥", "WARN": "⚠️", "ERR": "❌", "FIX": "🛡️"}.get(level, "•")
    print(f"[{ts}] [Iter {iteration}] {prefix} {msg}")

# ============================================================
# TASK 5: AGENT WITH FAILURE MODE HANDLING
# ============================================================
def run_agent_with_guardrails(user_query: str, max_iterations: int = 8) -> str:
    """
    Task 5 demo: Agent with failure mode detection and mitigations.
    Demonstrates: max_iterations, tool validation, error detection, graceful degradation.
    """
    messages = [{"role": "user", "content": user_query}]
    iteration = 0
    
    # Working memory for failure tracking
    failure_log = []
    tool_call_count = 0
    
    print(f"\n{'='*60}")
    print(f"TASK 5 DEMO: Failure Modes & Guardrails")
    print(f"Query: {user_query}")
    print(f"Max iterations: {max_iterations}")
    print(f"{'='*60}\n")
    
    while iteration < max_iterations:
        iteration += 1
        log(f"Starting iteration {iteration}/{max_iterations}", "INFO", iteration)
        
        # 1. CALL LLM
        try:
            response = client.messages.create(
                model="claude-3-5-sonnet-20241022",
                max_tokens=1024,
                messages=messages,
                tools=ALL_TOOLS,
                tool_choice={"type": "auto"}
            )
        except Exception as e:
            log(f"API call failed: {e}", "ERR", iteration)
            return f"API Error: {e}"
        
        tool_blocks = [b for b in response.content if b.type == "tool_use"]
        text_blocks = [b for b in response.content if b.type == "text"]
        
        # Log reasoning
        for block in text_blocks:
            log(f"REASONING: {block.text.strip()}", "INFO", iteration)
        
        # 2. NO TOOL CALLS = FINAL ANSWER
        if not tool_blocks:
            final = " ".join(b.text for b in text_blocks)
            log(f"FINAL ANSWER: {final}", "INFO", iteration)
            print(f"\n{'='*60}")
            print(f"✅ COMPLETED | Iterations: {iteration} | Tool calls: {tool_call_count}")
            if failure_log:
                print(f"⚠️  Failures handled: {len(failure_log)}")
                for f in failure_log:
                    print(f"   - {f}")
            print(f"{'='*60}\n")
            return final
        
        # 3. PROCESS TOOL CALLS WITH GUARDRAILS
        tool_results = []
        for block in tool_blocks:
            tool_call_count += 1
            log(f"TOOL CALL: {block.name}({json.dumps(block.input)})", "TOOL", iteration)
            
            # GUARDRAIL 1: Validate tool exists (prevents hallucinated tools)
            valid, err_msg = validate_tool_call(block.name, block.input)
            if not valid:
                log(f"VALIDATION FAILED: {err_msg}", "ERR", iteration)
                failure_log.append(f"Iter {iteration}: {err_msg}")
                # Return error as tool_result so LLM knows
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": f"Error: {err_msg}"
                })
                continue
            
            # GUARDRAIL 2: Execute tool
            executor = TOOL_EXECUTORS.get(block.name)
            if not executor:
                err = f"No executor for tool: {block.name}"
                log(err, "ERR", iteration)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {err}"})
                continue
            
            try:
                result = executor(block.input)
            except Exception as e:
                result = f"Error: Tool execution failed: {e}"
            
            log(f"RESULT: {result[:150]}", "OBS", iteration)
            
            # GUARDRAIL 3: Detect silent errors in result
            if is_error_result(result):
                log(f"SILENT ERROR DETECTED in tool result", "WARN", iteration)
                failure_log.append(f"Iter {iteration}: Tool '{block.name}' returned error: {result[:100]}")
                # Don't stop — let LLM see the error and decide
            
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": result
            })
        
        # 4. UPDATE CONVERSATION
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": tool_results})
    
    # GUARDRAIL 4: Max iterations reached (prevents infinite loops)
    log(f"MAX ITERATIONS ({max_iterations}) REACHED", "WARN", iteration)
    failure_log.append(f"Max iterations ({max_iterations}) reached without final answer")
    return f"Stopped after {max_iterations} iterations. Failures: {failure_log}"

# ============================================================
# TEST CASES FOR EACH FAILURE MODE
# ============================================================
if __name__ == "__main__":
    print("=" * 60)
    print("TEST 1: Normal multi-step (should work)")
    print("=" * 60)
    run_agent_with_guardrails("What's 15 + 27?")
    
    print("\n" + "=" * 60)
    print("TEST 2: Silent error from tool (broken_calculator)")
    print("=" * 60)
    run_agent_with_guardrails("Use broken_calculator to add 10 and 5")
    
    print("\n" + "=" * 60)
    print("TEST 3: Hallucinated tool (search_web doesn't exist)")
    print("=" * 60)
    run_agent_with_guardrails("Search the web for the weather in Tokyo")
    
    print("\n" + "=" * 60)
    print("TEST 4: Wrong arguments (multiply not supported)")
    print("=" * 60)
    run_agent_with_guardrails("Calculate 6 multiplied by 7")
    
    print("\n" + "=" * 60)
    print("TEST 5: Ambiguous request (may hit max_iterations)")
    print("=" * 60)
    run_agent_with_guardrails("Figure it out")


    

 Why Frameworks Exist (LangChain, LangGraph, CrewAI)
You just built a raw agent in ~100 lines of Python. So why do frameworks exist?

Frameworks don't add magic — they add guardrails and structure for the failures you just documented.

LangGraph replaces your while loop with a state graph: explicit nodes, edges, conditional routing, built-in max_iterations, checkpointing, and human-in-the-loop interrupts.
LangChain provides validated tool schemas (Pydantic), automatic retry logic, streaming, token counting, and integrations (vector stores, SQL, APIs) so you don't write boilerplate.
CrewAI adds role-based agents, task delegation, and multi-agent orchestration — things your single-loop agent can't do.
The raw agent teaches you what happens. Frameworks handle how to make it reliable at scale. You now know what the framework is doing under the hood: validating tools, catching errors, managing state, preventing infinite loops. When it breaks, you'll know why.